# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Sinh viên: Vũ Đức Thiện (2A202602437). Code nằm trong `submissions/2A202602437_vu_duc_thien/code/`:
`dataset.py`, `model.py`, `losses.py`, `train.py` (một hàm `run(Config)` cho mọi thí nghiệm), `inference.py`,
`benchmark.py`, `checks.py` (EDA, kiểm tra pipeline), `experiments.py` (Bước 3, 4), `results.py` (xlsx),
`test_selfcheck.py` (kiểm tra tự viết).

**Cách chạy:** chạy lần lượt từ trên xuống. Mọi kết quả ghi thẳng lên Google Drive (`MyDrive/K4-Day2-2A202602437`).
Phiên Colab bị ngắt thì chạy lại từ đầu: lần chạy đã xong (có `result.json`) được bỏ qua, quyết định đã chốt
(`tables/decisions.json`) không bị tính lại, và test không bao giờ chạy lần thứ hai.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.
Các ô có huấn luyện được đánh dấu **⚠️ CÓ HUẤN LUYỆN**.

## 0. Cài đặt môi trường

In [ ]:
# Chạy được trên Colab và Kaggle. Ô này clone fork của bạn (chạy lại thì chỉ git pull để lấy code mới).
# Colab: gắn Google Drive và trỏ runs/, predictions/, curves/, eval_out/ sang Drive để Colab ngắt không mất kết quả.
import os, sys
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
WORK_DIR = "/content" if IN_COLAB else "/kaggle/working" if IN_KAGGLE else os.getcwd()
os.chdir(WORK_DIR)

REPO_URL = "https://github.com/vuthien3002-sys/K4-Day2-VuDucThien-2A202602437.git"
REPO_DIR = f"{WORK_DIR}/K4-Day2-VuDucThien-2A202602437"
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log --oneline -1

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_DIR = "/content/drive/MyDrive/K4-Day2-2A202602437"
    for d in ["runs", "predictions", "curves", "eval_out", "figures", "tables"]:
        os.makedirs(f"{PERSIST_DIR}/{d}", exist_ok=True)
        if not os.path.lexists(d):
            os.symlink(f"{PERSIST_DIR}/{d}", d)
else:
    PERSIST_DIR = WORK_DIR   # Kaggle: /kaggle/working được lưu khi Save Version
# Checkpoint TRONG lúc train ghi ra đĩa cục bộ (nhanh, không làm đầy Drive bằng các phiên bản cũ);
# xong mỗi lần chạy, best.pt được chép một lần lên runs/ (Drive).
CKPT_DIR = "/content/ckpt" if IN_COLAB else "/tmp/ckpt"
print("WORK_DIR =", WORK_DIR, "| kết quả lưu ở", PERSIST_DIR)

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# REPO_DIR đặt ở ô trên. CODE_DIR là thư mục bài làm của bạn (chép từ starter/).
CODE_DIR = f"{REPO_DIR}/submissions/2A202602437_vu_duc_thien/code"
sys.path.insert(0, REPO_DIR)    # để import eval.py
sys.path.insert(0, CODE_DIR)    # dataset.py, model.py, train.py... của bạn

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
assert torch.cuda.is_available(), "CHƯA BẬT GPU: Thời gian chạy -> Thay đổi loại thời gian chạy -> T4 GPU -> Lưu, rồi chạy lại từ ô 1"

import json, time
from pathlib import Path
from IPython.display import Image as ShowImage, display
import dataset as D, model as M, losses as L, inference as I, benchmark as B
import checks as C, experiments as E, results as R
from train import Config, run, run_dir
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

### Tải dữ liệu (đã viết sẵn)
Nhãn và các file chia fold tải từ GitHub của tác giả. Ảnh: `images.zip` từ Zenodo (~490 MB), kiểm MD5 rồi giải nén.
Trên Colab, zip được cache vào Drive sau lần tải đầu, các phiên sau chỉ chép lại từ Drive.
Trên Kaggle, nếu đã **Add Input** dataset chứa 17.509 ảnh thì dùng luôn (không có zip để kiểm MD5, thay bằng kiểm tra tên file ở ô sau).

In [ ]:
import glob, hashlib

DATA_DIR = f"{WORK_DIR}/data" if IN_COLAB else "/tmp/deepweeds"   # Kaggle: ngoài /kaggle/working
os.makedirs(f"{DATA_DIR}/labels", exist_ok=True)

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O {DATA_DIR}/labels/{name}.csv {BASE}/{name}.csv
    assert os.path.getsize(f"{DATA_DIR}/labels/{name}.csv") > 0, "tải nhãn thất bại: kiểm tra Internet"
!ls -la {DATA_DIR}/labels

input_jpgs = glob.glob("/kaggle/input/**/*.jpg", recursive=True) if IN_KAGGLE else []
if input_jpgs:
    IMAGES_DIR = os.path.dirname(input_jpgs[0])
    print("Dùng ảnh từ Kaggle Input:", IMAGES_DIR, "(đã giải nén sẵn, không có zip để kiểm MD5)")
else:
    ZIP = f"{DATA_DIR}/images.zip"
    ZIP_CACHE = f"{PERSIST_DIR}/images.zip" if IN_COLAB else None
    if not os.path.exists(ZIP):
        if ZIP_CACHE and os.path.exists(ZIP_CACHE):
            !cp {ZIP_CACHE} {ZIP}
        else:
            !wget -q -O {ZIP} "https://zenodo.org/records/7939060/files/images.zip?download=1"
    h = hashlib.md5()
    with open(ZIP, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()} (xoá {ZIP} rồi chạy lại ô)"
    print("MD5 OK")
    if ZIP_CACHE and not os.path.exists(ZIP_CACHE):
        !cp {ZIP} {ZIP_CACHE}
    !unzip -q -n {ZIP} -d {DATA_DIR}/images
    IMAGES_DIR = os.path.dirname(glob.glob(f"{DATA_DIR}/images/**/*.jpg", recursive=True)[0])
!ls {IMAGES_DIR} | head

In [ ]:
# Kiểm tra ảnh khớp nhãn: đủ 17.509 dòng và mọi Filename trong labels.csv đều có trong IMAGES_DIR.
# Khi tạo Config ở các bước sau, luôn truyền images_dir=IMAGES_DIR, labels_dir=LABELS_DIR.
LABELS_DIR = f"{DATA_DIR}/labels"
labels_df = pd.read_csv(f"{LABELS_DIR}/labels.csv")
missing = set(labels_df["Filename"]) - set(os.listdir(IMAGES_DIR))
print("IMAGES_DIR =", IMAGES_DIR, "|", len(labels_df), "dòng labels.csv | thiếu", len(missing), "ảnh")
assert len(labels_df) == 17509 and not missing, f"thiếu ảnh, ví dụ: {sorted(missing)[:5]}"

In [ ]:
# Cấu hình chung cho MỌI lần chạy, và cơ chế "chốt quyết định": một lựa chọn (backbone, công thức, cách
# suy luận) được ghi vào tables/decisions.json kèm thời điểm, chạy lại notebook không tính lại nữa.
COMMON = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, ckpt_dir=CKPT_DIR, num_workers=2)

def make_cfg(**kw):
    return Config(**{**COMMON, **kw})

DECISIONS = Path("tables/decisions.json")

def decide(key, compute):
    d = json.loads(DECISIONS.read_text(encoding="utf-8")) if DECISIONS.exists() else {}
    if key not in d:
        d[key] = {"value": compute(), "decided_at": time.strftime("%Y-%m-%d %H:%M:%S")}
        DECISIONS.parent.mkdir(exist_ok=True)
        DECISIONS.write_text(json.dumps(d, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"{key} = {d[key]['value']}  (chốt lúc {d[key]['decided_at']})")
    return d[key]["value"]

def show(*paths):
    for p in paths:
        display(ShowImage(filename=str(p)))

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
# Kiểm tra chia dữ liệu fold 0 (S1-S4) và EDA: số ảnh mỗi tập/lớp, giao rỗng, hợp đủ 17.509, file tồn tại
train_df, val_df, test_df = D.load_split(LABELS_DIR, fold=0)
split = D.check_split(train_df, val_df, test_df, IMAGES_DIR)
eda = C.eda_table(split["per_class"])
eda.to_csv("tables/eda_split_counts.csv")
display(eda.round(2))
print(C.plot_class_distribution(split["per_class"], "figures/eda_class_distribution.png"))
C.plot_samples(train_df, IMAGES_DIR, "figures/eda_samples.png", n_per_class=4)
stats = C.image_stats(train_df, IMAGES_DIR, n=500)
json.dump({k: v for k, v in split.items() if k != "per_class"} | {"image_stats": stats},
          open("tables/eda_summary.json", "w"), indent=2, ensure_ascii=False)
print(stats)
show("figures/eda_class_distribution.png", "figures/eda_samples.png")

In [ ]:
# Kiểm tra pipeline KHÔNG huấn luyện: ảnh sau augmentation (đã giải chuẩn hoá) khớp nhãn, CutMix/Mixup trộn
# cả nhãn, loss ban đầu của head mới ~ ln 9, chế độ train/eval đúng lúc. Backbone mốc: ResNet-50.
BASE_BACKBONE = "resnet50.a1_in1k"
net = M.build_model(BASE_BACKBONE, pretrained=True)
mean, std = net.pretrained_cfg["mean"], net.pretrained_cfg["std"]
for aug in D.AUGS:
    C.plot_augmented(train_df, IMAGES_DIR, aug, f"figures/check_aug_{aug}.png", mean=mean, std=std)
for mode in ("cutmix", "mixup"):
    C.plot_mix(train_df, IMAGES_DIR, mode, f"figures/check_{mode}.png", mean=mean, std=std)
device = torch.device("cuda")
val_loader = D.make_loader(val_df, IMAGES_DIR, D.build_transforms(False, 224, mean=mean, std=std), 64, False)
init = C.initial_loss(net.to(device), val_loader, device, n_batches=10)
modes = C.check_modes("resnet18")
print("Loss ban đầu:", init, "\nChế độ train/eval:", modes)
json.dump({"initial_loss": init, "modes": modes}, open("tables/check_pipeline.json", "w"), indent=2)
show("figures/check_aug_basic.png", "figures/check_aug_trivial.png", "figures/check_cutmix.png")

In [ ]:
# Kiểm tra tự viết (focal gamma=0 == CE, CutMix theo diện tích thật, gộp BN, nhóm tham số, EMA...) trên CPU
!cd {CODE_DIR} && DEEPWEEDS_DIR={DATA_DIR} python -m unittest test_selfcheck 2>&1 | tail -4
# Bộ test của repo (eval.py và khung starter/)
!cd {REPO_DIR} && python -m unittest discover -s tests 2>&1 | tail -3

In [ ]:
# ⚠️ CÓ HUẤN LUYỆN (150 bước trên MỘT batch): quá khớp 36 ảnh train cố định (không augmentation)
# tới loss gần 0. Không làm được thì lỗi nằm ở code/model, chưa chạy tiếp.
torch.manual_seed(0)
idx = train_df.groupby("Label").head(4).index  # 36 ảnh: 4 ảnh mỗi lớp, đủ 9 lớp
tf = D.build_transforms(False, 224, mean=mean, std=std)
ds = D.DeepWeedsDataset(train_df.loc[idx], IMAGES_DIR, tf)
xb = torch.stack([ds[i][0] for i in range(len(ds))]); yb = torch.tensor([ds[i][1] for i in range(len(ds))])
over = C.overfit_one_batch(M.build_model(BASE_BACKBONE, pretrained=True), xb, yb, device, steps=150, lr=3e-4,
                           path="figures/check_overfit_one_batch.png")
print(over)
json.dump(over, open("tables/check_overfit.json", "w"), indent=2)
show("figures/check_overfit_one_batch.png")
assert over["final_loss"] < 0.05 and over["batch_acc_eval"] == 1.0, "Không overfit được 1 batch: dừng, kiểm tra code"

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4: AdamW, LR backbone 1e-4 / head 1e-3, wd 0,05 trừ norm/bias, warmup 1 epoch
+ cosine, CE, batch 64, 12 epoch, AMP, chọn checkpoint theo macro-F1 val), cùng seed 0, mỗi backbone một lần chạy.
Tag trọng số đều là ImageNet-1k để so sánh công bằng. Đủ ràng buộc: ResNet (B01), ConvNeXt (B02),
transformer (B03 DeiT-S, B04 Swin-T), mạng nhẹ (B05 EfficientNet-B0, B06 MobileNetV3-L).

In [ ]:
# ⚠️ CÓ HUẤN LUYỆN: 6 lần chạy (mỗi lần 12 epoch). Lần đã xong được bỏ qua khi chạy lại.
BACKBONES = [
    ("B01", "resnet50.a1_in1k", "resnet50"),
    ("B02", "convnext_tiny.fb_in1k", "convnext_tiny"),
    ("B03", "deit_small_patch16_224.fb_in1k", "deit_small"),
    ("B04", "swin_tiny_patch4_window7_224.ms_in1k", "swin_tiny"),
    ("B05", "efficientnet_b0.ra_in1k", "efficientnet_b0"),
    ("B06", "mobilenetv3_large_100.ra_in1k", "mobilenetv3_large"),
]
BACKBONES_BY_ID = {e: {"exp_id": e, "backbone": b, "desc": d} for e, b, d in BACKBONES}
for exp_id, backbone, desc in BACKBONES:
    run(make_cfg(exp_id=exp_id, backbone=backbone, desc=desc, seed=0))

In [ ]:
# Bảng Backbones, biểu đồ F1 theo độ trễ / GMAC, và chọn backbone đi tiếp (dựa trên số liệu val)
res = R.load_results()
bs = R.backbones_sheet(res)
display(bs.round(4))
show(R.plot_backbones(bs, "figures/backbones_tradeoff.png"))

def pick_backbone():
    # Macro-F1 val cao nhất. Nếu một backbone nhanh hơn ít nhất 2 lần mà F1 kém không quá 0,005 thì ưu tiên nó
    # (chênh lệch nhỏ như vậy với 1 seed chưa phân biệt được).
    best = bs.loc[bs["macro-F1 val"].idxmax()]
    lat = "độ trễ batch-1 fp32 p50 (ms)"
    near = bs[(bs["macro-F1 val"] >= best["macro-F1 val"] - 0.005) & (bs[lat] * 2 <= best[lat])]
    row = near.sort_values("macro-F1 val", ascending=False).iloc[0] if len(near) else best
    return BACKBONES_BY_ID[row["exp_id"]]

CHOSEN = decide("backbone_for_steps_2_to_4", pick_backbone)
CHOSEN_BB = CHOSEN["backbone"]

In [ ]:
# Ngân sách GPU (GUIDE mục 7): thời gian/epoch ĐO ĐƯỢC ở Bước 1 x số epoch x số lần chạy còn lại
sec_epoch = float(res.loc[res["exp_id"] == CHOSEN["exp_id"], "train_s_per_epoch"].iloc[0])
plan = {"Bước 2: T00 x 3 seed": 3, "Bước 2: ablation 1 seed": 11, "Bước 2: kết hợp": 1, "Bước 4: F01 x 3 seed": 3}
hours = {k: n * 12 * sec_epoch / 3600 for k, n in plan.items()}
print(f"{CHOSEN_BB}: {sec_epoch:.0f}s/epoch (train) -> ước lượng còn lại:")
for k, h in hours.items():
    print(f"  {k:28s} {h:5.2f} giờ")
print(f"  {'TỔNG':28s} {sum(hours.values()):5.2f} giờ GPU (chưa tính đánh giá val, lần chạy hỏng)")
print("Kế hoạch: ablation Bước 2 chỉ trên 1 backbone (đã chọn), 1 seed; dành 3 seed cho T00 và chung kết.")
json.dump({"backbone": CHOSEN_BB, "sec_per_epoch": sec_epoch, "plan_hours": hours},
          open("tables/gpu_budget.json", "w"), indent=2, ensure_ascii=False)

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố** (cùng backbone đã chọn, cùng seed 0, cùng 12 epoch). `T00` chạy 3 seed để
đo nhiễu do seed (std) — dùng để kết luận Δ có vượt nhiễu không, và cũng là mốc của Bước 4.
Thứ tự chạy theo ưu tiên (loss, augmentation, khởi tạo, EMA, sampler, Mixup) để nếu hết GPU thì phần quan trọng đã xong.

In [ ]:
# ⚠️ CÓ HUẤN LUYỆN: T00 x 3 seed + 11 ablation (1 seed). Lần đã xong được bỏ qua khi chạy lại.
ABLATIONS = {  # exp_id: (trục, khác T00 ở điểm nào, ghi đè Config)
    "T01": ("A. khởi tạo", "từ đầu (không tiền huấn luyện)", dict(init="scratch")),
    "T02": ("A. khởi tạo", "đóng băng backbone, chỉ train head", dict(init="frozen")),
    "T03": ("B. augmentation", "+ ColorJitter", dict(aug="color")),
    "T04": ("B. augmentation", "+ TrivialAugmentWide", dict(aug="trivial")),
    "T05": ("B. augmentation", "+ CutMix (alpha 1,0)", dict(mix="cutmix", mix_alpha=1.0)),
    "T06": ("B. augmentation", "+ Mixup (alpha 0,2)", dict(mix="mixup", mix_alpha=0.2)),
    "T07": ("C. loss", "label smoothing 0,1", dict(loss="ls", label_smoothing=0.1)),
    "T08": ("C. loss", "focal loss gamma 2", dict(loss="focal", focal_gamma=2.0)),
    "T09": ("C. loss", "CE trọng số class-balanced (beta 0,999)", dict(loss="ce_weighted", class_weight_beta=0.999)),
    "T10": ("D. cân bằng mẫu", "sampler cân bằng lớp", dict(sampler="balanced")),
    "T11": ("F. chính quy hoá", "EMA trọng số (decay 0,998)", dict(ema_decay=0.998)),
}
DESC = {"T01": "scratch", "T02": "frozen", "T03": "color", "T04": "trivialaug", "T05": "cutmix", "T06": "mixup",
        "T07": "label_smoothing", "T08": "focal", "T09": "class_balanced_ce", "T10": "balanced_sampler", "T11": "ema"}
PRIORITY = ["T07", "T08", "T09", "T03", "T04", "T05", "T01", "T02", "T11", "T10", "T06"]
for seed in (0, 1, 2):
    run(make_cfg(exp_id="T00", backbone=CHOSEN_BB, desc="baseline", seed=seed))
for exp_id in PRIORITY:
    run(make_cfg(exp_id=exp_id, backbone=CHOSEN_BB, desc=DESC[exp_id], seed=0, **ABLATIONS[exp_id][2]))

In [ ]:
# Bảng ablation: Δ so với T00 cùng seed, so với nhiễu (std của T00 qua 3 seed). Chọn yếu tố để kết hợp.
res = R.load_results()
design = {"T00": ("nền", "công thức nền")} | {k: v[:2] for k, v in ABLATIONS.items()}
tr = R.training_sheet(res, design)
NOISE = R.noise_std(res, "T00")
display(tr.round(4))
print(f"Nhiễu do seed (std macro-F1 val của T00, 3 seed) = {NOISE:.4f}")

def pick_combo():
    # Tham lam theo trục: mỗi trục lấy giá trị có Δ lớn nhất, giữ các trục có Δ > nhiễu. CE có trọng số lớp (T09)
    # và sampler cân bằng (T10) cùng xử lý mất cân bằng lớp: ghép cả hai là cân bằng hai lần, nên chỉ giữ cái tốt hơn.
    # Nếu ít hơn 2 trục vượt nhiễu, vẫn kết hợp 2 trục có Δ lớn nhất (ghi rõ là chưa vượt nhiễu) để kiểm tra
    # hiệu ứng cộng dồn hay triệt tiêu (GUIDE mục 3.1, ý 4).
    t = tr[(tr["exp_id"] != "T00") & (tr["seed"] == 0)].sort_values("Δ vs T00 (macro-F1)", ascending=False)
    best_per_axis = t.groupby("trục").head(1)
    imbalance = best_per_axis[best_per_axis["exp_id"].isin(["T09", "T10"])]
    if len(imbalance) == 2:
        best_per_axis = best_per_axis.drop(imbalance.index[1])
    win = best_per_axis[best_per_axis["Δ vs T00 (macro-F1)"] > NOISE]
    if len(win) < 2:
        win = best_per_axis.head(2)
    return sorted(win["exp_id"].tolist())

COMBO = decide("combo_factors_T12", pick_combo)
COMBO_OVERRIDES = {k: v for e in COMBO for k, v in ABLATIONS[e][2].items()}
print("T12 = T00 +", [ABLATIONS[e][1] for e in COMBO], "->", COMBO_OVERRIDES)

In [ ]:
# ⚠️ CÓ HUẤN LUYỆN: kết hợp các yếu tố tốt (T12), 1 seed
if COMBO:
    run(make_cfg(exp_id="T12", backbone=CHOSEN_BB, desc="combo", seed=0, **COMBO_OVERRIDES))
res = R.load_results()
design["T12"] = ("kết hợp", " + ".join(ABLATIONS[e][1] for e in COMBO))
tr = R.training_sheet(res, design)
display(tr.round(4))

def pick_recipe():
    # Chỉ đổi khỏi công thức nền khi một cấu hình (seed 0) vượt T00 cùng seed HƠN NHIỄU. Chênh lệch nhỏ hơn nhiễu là
    # "không phân biệt được" (GUIDE N4), nên giữ T00.
    best = tr[(tr["seed"] == 0) & (tr["exp_id"] != "T00")].sort_values("macro-F1 val", ascending=False).iloc[0]
    return best["exp_id"] if best["Δ vs T00 (macro-F1)"] > NOISE else "T00"

# Lượt đầu (khoá "final_recipe" trong decisions.json) chọn T07 theo macro-F1 cao nhất mà KHÔNG so với nhiễu. F01 = T07 x 3
# seed cho macro-F1 val thấp hơn T00 (xem ô ⛔), nên trước khi chạy test đã sửa quy tắc như trên (khoá "final_recipe_v2").
RECIPE_EXP = decide("final_recipe_v2", pick_recipe)
RECIPE = {} if RECIPE_EXP == "T00" else (COMBO_OVERRIDES if RECIPE_EXP == "T12" else ABLATIONS[RECIPE_EXP][2])
print("Công thức chung kết = T00 +", RECIPE)

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**, không huấn luyện lại: nạp `best.pt` của công thức đã chốt (seed 0). I00 1 view (mốc) · I01 TTA lật ·
I02 5-crop / 10-crop / 3 tỉ lệ · I03 gộp logit so với gộp xác suất · I04 dò độ phân giải · I05 ensemble (3 seed T00;
3 backbone tốt nhất) · I06 EMA · I07 temperature scaling + ECE · I08 gộp BN / FP16. Độ trễ đo bằng `benchmark.py`:
warmup 10, `cuda.synchronize`, 100 lần, p50/p95/p99, không tính tiền xử lý.

In [ ]:
# Không huấn luyện: chỉ suy luận trên val và đo độ trễ (vài phút), cho mô hình của công thức đã chốt (seed 0)
recipe_cfg = make_cfg(exp_id=RECIPE_EXP, backbone=CHOSEN_BB, seed=0, **RECIPE)
top3 = bs.sort_values("macro-F1 val", ascending=False)["exp_id"].head(3)
ensembles = {
    "I05_3seeds_T00": [make_cfg(exp_id="T00", backbone=CHOSEN_BB, seed=s) for s in (0, 1, 2)],
    "I05_top3_backbones": [make_cfg(exp_id=e, backbone=BACKBONES_BY_ID[e]["backbone"], seed=0) for e in top3],
}
ema_pair = (make_cfg(exp_id="T00", backbone=CHOSEN_BB, seed=0),
            make_cfg(exp_id="T11", backbone=CHOSEN_BB, seed=0, ema_decay=0.998))
inf_file, lat_file = Path(f"tables/inference_val_{RECIPE_EXP}.csv"), Path(f"tables/latency_{RECIPE_EXP}.csv")
if not inf_file.exists():
    inf, lat = E.inference_suite(recipe_cfg, torch.device("cuda"), ensembles=ensembles, ema_pair=ema_pair)
    inf.to_csv(inf_file, index=False); lat.to_csv(lat_file, index=False)
inf, lat = pd.read_csv(inf_file), pd.read_csv(lat_file)
# Lượt Bước 3 đầu tiên chạy trên T07 (trước khi sửa quy tắc chọn công thức): giữ lại, cùng đưa vào results.xlsx
old_inf, old_lat = Path("tables/inference_val.csv"), Path("tables/latency.csv")
inf_all = pd.concat([inf] + ([pd.read_csv(old_inf)] if old_inf.exists() and RECIPE_EXP != "T07" else []), ignore_index=True)
lat_all = pd.concat([lat] + ([pd.read_csv(old_lat)] if old_lat.exists() and RECIPE_EXP != "T07" else []), ignore_index=True)
display(R.inference_sheet(inf).round(4)); display(lat.round(3))
show(E.plot_tradeoff(inf, "figures/inference_tradeoff.png", f"Bước 3 trên val: {RECIPE_EXP} ({CHOSEN['desc']})"))

In [ ]:
# Chọn cách suy luận cho chung kết trên VAL: cách rẻ nhất trong các cách có macro-F1 cách tốt nhất <= nhiễu.
# Khoá "_v2": chọn lại trên mô hình của công thức đã chốt ở trên (khoá cũ ứng với lượt trên T07).
def pick_method():
    row = E.choose_method(inf, NOISE)
    return {"exp_id": row["exp_id"], "method": row["method"], "space": row["space"]}

METHOD = decide("final_inference_method_v2", pick_method)

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Cấu hình chốt trên **val** (`tables/decisions.json`): backbone, công thức, cách suy luận + temperature scaling (T khớp trên val).
2. `F01` (lần đầu) = công thức T07 (label smoothing), 3 seed. Lợi thế của T07 lúc chọn (1 seed) nhỏ hơn nhiễu, và với 3 seed
   macro-F1 val của F01 thấp hơn T00 (số liệu ở ô ⛔). F01 bị loại **trên val, trước khi chạy test**; quy tắc chọn công thức
   được sửa để tính đến nhiễu (ô cuối Bước 2).
3. `F02` (chung kết) = công thức đã chốt. Nếu trùng công thức nền T00 thì dùng lại 3 lần chạy T00 (seed 0/1/2), không train
   lại; khác biệt so với mốc nằm ở cách suy luận. Mốc = `T00` + `I00` (1 view), cùng 3 seed.
4. Test chạy **đúng một lần mỗi seed** cho mỗi cấu hình: `final_predictions` và `run(..., save_test_predictions=True)` từ chối chạy lại.

In [ ]:
# Chung kết F02. Công thức chốt là T00: dùng lại 3 lần chạy T00 (không train). Nếu khác T00: ⚠️ train 3 seed (không đụng test).
FINAL = "F02"
FINAL_SEEDS = (0, 1, 2)
if RECIPE_EXP == "T00":
    f_cfgs = [make_cfg(exp_id="T00", backbone=CHOSEN_BB, desc="baseline", seed=s) for s in FINAL_SEEDS]
else:
    f_cfgs = [make_cfg(exp_id=FINAL, backbone=CHOSEN_BB, desc="final", seed=s, **RECIPE) for s in FINAL_SEEDS]
for c in f_cfgs:
    run(c)  # lần chạy đã xong thì bỏ qua

In [ ]:
# ⛔ ĐIỂM DỪNG TRƯỚC TEST. "Chạy tất cả" dừng ở đây để xem lại các quyết định trên val trước khi chạm vào test.
# Tiếp tục: bấm vào ô TEST ngay bên dưới -> Thời gian chạy -> "Chạy ô đã chọn và các ô bên dưới" (Ctrl+F10).
print(json.dumps(json.loads(DECISIONS.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))
res = R.load_results()
display(res[res["exp_id"].isin(["T00", "T07", "F01", RECIPE_EXP])][["exp_id", "seed", "gpu", "val_macro_f1", "val_top1",
                                                                  "val_recall_chinee", "val_recall_snake", "best_epoch"]])
for e in ("T00", "F01"):
    v = res.loc[res["exp_id"] == e, "val_macro_f1"]
    if len(v) > 1:
        print(f"{e}: macro-F1 val {v.mean():.4f} ± {v.std(ddof=1):.4f} ({len(v)} seed)")
print(f"Chung kết {FINAL} = công thức {RECIPE_EXP} + suy luận {METHOD} + temperature scaling")
if not Path(f"predictions/{FINAL}_seed0_test.csv").exists():
    raise RuntimeError("Dừng trước Bước 4 (test). Gửi ảnh kết quả ở trên, rồi chạy tiếp từ ô TEST bên dưới.")

In [ ]:
# TEST — mỗi seed đúng MỘT lần. F02: cách suy luận đã chốt + T khớp trên val (ghi cả bản chưa hiệu chuẩn F02uncal);
# mốc T00: 1 view. Chạy lại ô này không chạy test lần nữa.
for c in f_cfgs:
    if not Path(f"predictions/{FINAL}_seed{c.seed}_test.csv").exists():
        E.final_predictions(c, METHOD["method"], METHOD["space"], FINAL)
for s in FINAL_SEEDS:
    run(make_cfg(exp_id="T00", backbone=CHOSEN_BB, desc="baseline", seed=s, save_test_predictions=True))
!ls predictions | grep -E "^({FINAL}|{FINAL}uncal|T00)_seed[0-9]_test.csv$"

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/{FINAL}_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag {FINAL} --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/{FINAL}uncal_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag {FINAL}uncal --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). p95 batch 1 của cấu hình chung kết (đo đúng cách).
final_info = json.loads((run_dir(f_cfgs[0]) / f"{FINAL}_final.json").read_text(encoding="utf-8"))
P95 = final_info["latency_b1_fp32"]["p95"]
print(f"Cấu hình thời gian thực = chung kết ({METHOD}), p95 batch 1 fp32 = {P95:.2f} ms trên {torch.cuda.get_device_name(0)}")
!python {REPO_DIR}/eval.py grade --final "predictions/{FINAL}_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" --uncal "predictions/{FINAL}uncal_seed*_test.csv" \
    --final-val "predictions/{FINAL}_seed*_val.csv" --latency-p95-ms {P95:.2f} --latency-method proper \
    --test-csv {LABELS_DIR}/test_subset0.csv --val-csv {LABELS_DIR}/val_subset0.csv \
    --labels {LABELS_DIR}/labels.csv --out eval_out

In [ ]:
# Phân tích lỗi trên test (SAU khi đã chốt mọi thứ, không dùng để chọn gì): ma trận nhầm lẫn (tổng 3 seed) và
# ảnh bị nhầm giữa Chinee Apple (0) và Snake Weed (7)
from matplotlib.figure import Figure
cm = pd.read_csv(f"eval_out/{FINAL}_confusion_sum.csv", index_col=0)
display(cm)
fig = Figure(figsize=(8, 7), dpi=120); ax = fig.subplots()
norm = cm.values / cm.values.sum(1, keepdims=True)
ax.imshow(norm, cmap="Blues")
for i in range(9):
    for j in range(9):
        ax.text(j, i, int(cm.values[i, j]), ha="center", va="center", fontsize=7,
                color="white" if norm[i, j] > 0.5 else "black")
ax.set_xticks(range(9), D.CLASS_NAMES, rotation=45, ha="right"); ax.set_yticks(range(9), D.CLASS_NAMES)
ax.set(xlabel="dự đoán", ylabel="nhãn thật",
       title=f"{FINAL} trên test: ma trận nhầm lẫn (tổng 3 seed, màu = tỉ lệ theo hàng)")
fig.tight_layout(); fig.savefig("figures/final_confusion_matrix.png")
E.plot_errors(f"predictions/{FINAL}_seed0_test.csv", IMAGES_DIR, "figures/errors_chinee_as_snake.png", 0, 7)
E.plot_errors(f"predictions/{FINAL}_seed0_test.csv", IMAGES_DIR, "figures/errors_snake_as_chinee.png", 7, 0)
show("figures/final_confusion_matrix.png", "figures/errors_chinee_as_snake.png", "figures/errors_snake_as_chinee.png")

## Bước 5 — Sản phẩm
`results.xlsx` (7 sheet theo GUIDE mục 6.1), `curves/<exp_id>_*.png` (tự vẽ khi train), `figures/`, `tables/`,
`predictions/`, `eval_out/`. Ô cuối nén mọi thứ thành một file zip trên Drive để tải về đưa vào repo.

In [ ]:
res = R.load_results()
groups = {FINAL: (f"{CHOSEN['desc']} + {RECIPE_EXP} + {METHOD['method']} ({METHOD['space']}) + TS",
                  f"predictions/{FINAL}_seed*_test.csv"),
          "T00": (f"{CHOSEN['desc']} + T00 + I00 (mốc)", "predictions/T00_seed*_test.csv"),
          f"{FINAL}uncal": (f"{FINAL} chưa temperature scaling", f"predictions/{FINAL}uncal_seed*_test.csv")}
f_val = {FINAL: {c.seed: json.loads((run_dir(c) / f"{FINAL}_final.json").read_text(encoding="utf-8"))["val"]["macro_f1"]
                 for c in f_cfgs},
         "T00": res[res["exp_id"] == "T00"].set_index("seed")["val_macro_f1"].to_dict()}
test_csv = f"{LABELS_DIR}/test_subset0.csv"
sheets = {
    "Backbones": R.backbones_sheet(res),
    "Training": R.training_sheet(res, design),
    "Inference": R.inference_sheet(inf_all),
    "Final": R.final_sheet(groups, test_csv, f_val),
    "PerClass": R.perclass_sheet({FINAL: groups[FINAL], "T00": groups["T00"]}, test_csv),
    "Latency": lat_all,
    "Summary": R.summary_sheet(res, inf_all),
}
f01 = res.loc[res["exp_id"] == "F01", "val_macro_f1"]
notes = {"Training": f"Nhiễu do seed (std macro-F1 val của T00, 3 seed) = {NOISE:.4f}. Δ so với T00 cùng seed 0.",
         "Final": "Test chạy một lần mỗi seed; T (temperature) khớp trên val. "
                  + (f"F01 (T07, label smoothing, {len(f01)} seed, macro-F1 val {f01.mean():.4f} ± {f01.std(ddof=1):.4f}) "
                     "bị loại trên val trước khi chạy test." if len(f01) > 1 else ""),
         "Inference": "Lượt chính trên mô hình của công thức đã chốt; lượt đầu trên T07 giữ lại để truy vết (cột mô hình).",
         "Summary": "Top 10 theo macro-F1 val (lần chạy huấn luyện và cách suy luận)."}
XLSX = Path(PERSIST_DIR) / "results.xlsx"  # lưu thẳng lên Drive
print(R.write_xlsx(XLSX, sheets, notes))
for k, v in sheets.items():
    print(k, v.shape)

In [ ]:
# Nén sản phẩm để tải về (Drive: MyDrive/K4-Day2-2A202602437/submission_artifacts.zip). Không gồm checkpoint, ảnh dataset.
import zipfile
keep_pred = [p for p in Path("predictions").glob("*.csv") if p.name.startswith((f"{FINAL}_", f"{FINAL}uncal_", "T00_seed"))]
zpath = Path(PERSIST_DIR) / "submission_artifacts.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(XLSX, "results.xlsx")
    for folder in ("curves", "figures", "tables", "eval_out"):
        for p in Path(folder).glob("*"):
            z.write(p, f"{folder}/{p.name}")
    for p in keep_pred:
        z.write(p, f"predictions/{p.name}")
    for p in Path("runs").glob("*/seed*/*"):
        if p.suffix in (".json", ".csv"):
            z.write(p, str(p))
print(zpath, f"{zpath.stat().st_size / 1e6:.1f} MB")